# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.8 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID = 'task059'
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path('/mnt/data/task059.json')
KAGGLE_TASK_JSON = Path(COMPETITION) / f'{TASK_ID}.json'
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = Path.cwd() / f'{TASK_ID}_segment_lattice_max_cell_fill_onnx'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f'{TASK_ID}.onnx'
SUBMISSION_PATH = Path.cwd() / 'submission.zip'
SUMMARY_PATH = OUT_DIR / f'{TASK_ID}_validation_summary.json'
with TASK_JSON.open('r') as f:
    task=json.load(f)
print(TASK_ID, len(task.get('train', [])), len(task.get('test', [])), len(task.get('arc-gen', [])))

task059 3 1 262


In [6]:
def grid_to_tensor(grid, h=H, w=W, ch=CH, offset=(0,0)):
    x=np.zeros((1,ch,h,w),dtype=np.float32)
    ro,co=offset
    for r,row in enumerate(grid):
        for c,v in enumerate(row):
            rr,cc=r+ro,c+co
            if 0 <= rr < h and 0 <= cc < w:
                x[0,int(v),rr,cc]=1.0
    return x

def tensor_to_grid(y, h, w):
    return y[0,:,:h,:w].argmax(axis=0).astype(int).tolist()

def expected_tensor(ex):
    return grid_to_tensor(ex['output'])

In [7]:
class Task059SegmentLatticeMaxCellFill(nn.Module):
    """Separator-bounded segment-cell model for task059.

    Structural rule:
    - Active canvas is sum(input_channels) > 0, so padded zeros are not treated as color-0 background.
    - Color 5 separator rows/columns define a lattice over the active canvas.
    - For every non-separator pixel, compute its whole cell using a static row/column same-segment relation.
    - Count nonzero, non-separator markers in each cell.
    - Fill every cell whose count equals the global maximum count, using the marker color already present in that cell.
    - Keep separator pixels and force every pixel outside active_canvas to all-zero across all channels.

    This is fully static ONNX: no Loop/Scan/NonZero/Unique/Script/Function and no visible-shape export.
    """
    def __init__(self):
        super().__init__()
        between_rows=np.zeros((H*H,H),np.float32)
        between_cols=np.zeros((W*W,W),np.float32)
        for r in range(H):
            for q in range(H):
                lo=min(r,q)+1
                hi=max(r,q)
                if hi > lo:
                    between_rows[r*H+q, lo:hi]=1.0
        for c in range(W):
            for k in range(W):
                lo=min(c,k)+1
                hi=max(c,k)
                if hi > lo:
                    between_cols[c*W+k, lo:hi]=1.0
        self.register_buffer('between_rows_t', torch.from_numpy(between_rows.T.copy()))
        self.register_buffer('between_cols_t', torch.from_numpy(between_cols.T.copy()))

    def _cell_sum(self, same_r, same_c, field):
        # field: [B,1,30,30].  same_r/same_c: [B,30,30].
        tmp=torch.matmul(same_r, field[:,0,:,:])
        return torch.matmul(tmp, same_c.transpose(1,2)).unsqueeze(1)

    def forward(self, x):
        active=(x.sum(dim=1, keepdim=True)>0.5).float()
        sep=x[:,5:6]*active

        row_act=(active.sum(dim=3)>0.5).float()
        col_act=(active.sum(dim=2)>0.5).float()
        row_den=torch.clamp(active.sum(dim=3), min=1.0)
        col_den=torch.clamp(active.sum(dim=2), min=1.0)

        # Separator row/column = full active row/column of color 5.
        hsep=((sep.sum(dim=3)>row_den-0.5).float()*row_act)[:,0,:]
        vsep=((sep.sum(dim=2)>col_den-0.5).float()*col_act)[:,0,:]

        # row_blocked[r,q] is positive if a separator row lies strictly between r and q.
        row_blocked=torch.matmul(hsep, self.between_rows_t).reshape(-1,H,H)
        col_blocked=torch.matmul(vsep, self.between_cols_t).reshape(-1,W,W)

        row_not=(1.0-hsep)*row_act[:,0,:]
        col_not=(1.0-vsep)*col_act[:,0,:]
        same_r=(row_blocked<0.5).float()*row_not.unsqueeze(2)*row_not.unsqueeze(1)
        same_c=(col_blocked<0.5).float()*col_not.unsqueeze(2)*col_not.unsqueeze(1)

        interior=active*(1.0-sep)
        foreground=interior*(1.0-x[:,0:1])
        cell_count=self._cell_sum(same_r, same_c, foreground)*interior
        max_count=cell_count.amax(dim=(2,3), keepdim=True)
        selected=interior*(torch.abs(cell_count-max_count)<0.5).float()*(max_count>0.5).float()

        channels=[None]*10
        occ=sep.clone()
        for col in range(1,10):
            if col==5:
                out=sep
            else:
                col_count=self._cell_sum(same_r, same_c, interior*x[:,col:col+1])*interior
                out=selected*(col_count>0.5).float()
            channels[col]=out
            occ=torch.clamp(occ+out, 0, 1)
        channels[0]=active*(1.0-occ)
        return torch.cat(channels, dim=1)*active

model=Task059SegmentLatticeMaxCellFill().eval()


In [8]:
# Fast PyTorch sanity check before export.
with torch.no_grad():
    for split in ['train','test']:
        ok=0
        for ex in task.get(split, []):
            y=model(torch.from_numpy(grid_to_tensor(ex['input']))).numpy()
            ok += int(np.array_equal((y>0.5).astype(np.float32), expected_tensor(ex)))
        print(split, ok, '/', len(task.get(split, [])))

train 3 / 3
test 1 / 1


In [9]:
dummy = torch.from_numpy(grid_to_tensor(task['test'][0]['input']))
torch.onnx.export(
    model, dummy, str(ONNX_PATH), input_names=['input'], output_names=['output'],
    opset_version=17, do_constant_folding=True, dynamic_axes=None, dynamo=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))
print('ONNX:', ONNX_PATH, 'bytes:', ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/2058970932.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX: /kaggle/working/task059_segment_lattice_max_cell_fill_onnx/task059.onnx bytes: 140728


In [10]:
def vi_shape(vi):
    return [int(d.dim_value) if d.dim_value else (d.dim_param or None) for d in vi.type.tensor_type.shape.dim]
onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
summary_static = {
    'input_shape': vi_shape(onnx_model.graph.input[0]),
    'output_shape': vi_shape(onnx_model.graph.output[0]),
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'ops': dict(ops),
    'forbidden_ops': sorted(forbidden & set(ops)),
}
print(json.dumps(summary_static, indent=2)[:4000])
assert summary_static['input_shape'] == [1,10,30,30]
assert summary_static['output_shape'] == [1,10,30,30]
assert summary_static['onnx_size_bytes'] < 1_400_000
assert not summary_static['forbidden_ops']

{
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 140728,
  "ops": {
    "Identity": 1,
    "Constant": 101,
    "ReduceSum": 5,
    "Greater": 14,
    "Cast": 17,
    "Slice": 10,
    "Mul": 40,
    "Clip": 11,
    "Sub": 8,
    "Gather": 13,
    "MatMul": 20,
    "Reshape": 2,
    "Less": 3,
    "Unsqueeze": 13,
    "Transpose": 1,
    "ReduceMax": 1,
    "Abs": 1,
    "Add": 9,
    "Concat": 1
  },
  "forbidden_ops": []
}


In [11]:
sess_options = ort.SessionOptions(); sess_options.intra_op_num_threads=1; sess_options.inter_op_num_threads=1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=['CPUExecutionProvider'])

def validate_examples(examples):
    ok=0; bad=[]; outside_zero_ok=0; active_exact_ok=0
    for i,ex in enumerate(examples):
        x=grid_to_tensor(ex['input'])
        y=sess.run(None, {'input':x})[0]
        pred=(y>0.5).astype(np.float32)
        exp=expected_tensor(ex)
        if np.array_equal(pred, exp):
            ok += 1
        else:
            bad.append(i)
        active=(x.sum(axis=1, keepdims=True)>0.5).astype(np.float32)
        outside_zero_ok += bool(np.all(pred*(1-active)==0))
        active_exact_ok += bool(np.array_equal(pred*active, exp*active))
    return {'ok':ok, 'total':len(examples), 'bad_first10':bad[:10], 'outside_zero_ok':outside_zero_ok, 'active_exact_ok':active_exact_ok}

rng=random.Random(0)
inds=list(range(len(task.get('arc-gen', []))))
rng.shuffle(inds)
holdout=[task['arc-gen'][i] for i in inds[:max(1,int(0.60*len(inds)))]] if inds else []
rest=[task['arc-gen'][i] for i in inds[max(1,int(0.60*len(inds))):]] if inds else []
summary={
    'train': validate_examples(task.get('train', [])),
    'test': validate_examples(task.get('test', [])),
    'arc_gen_60_percent_holdout': validate_examples(holdout),
    'arc_gen_rest': validate_examples(rest),
    'arc_gen_full_diagnostic': validate_examples(task.get('arc-gen', [])),
}
print(json.dumps(summary, indent=2))
assert summary['train']['ok'] == summary['train']['total']
assert summary['test']['ok'] == summary['test']['total']
if holdout:
    assert summary['arc_gen_60_percent_holdout']['ok'] == summary['arc_gen_60_percent_holdout']['total']
assert summary['train']['outside_zero_ok'] == summary['train']['total']
assert summary['test']['outside_zero_ok'] == summary['test']['total']

{
  "train": {
    "ok": 3,
    "total": 3,
    "bad_first10": [],
    "outside_zero_ok": 3,
    "active_exact_ok": 3
  },
  "test": {
    "ok": 1,
    "total": 1,
    "bad_first10": [],
    "outside_zero_ok": 1,
    "active_exact_ok": 1
  },
  "arc_gen_60_percent_holdout": {
    "ok": 157,
    "total": 157,
    "bad_first10": [],
    "outside_zero_ok": 157,
    "active_exact_ok": 157
  },
  "arc_gen_rest": {
    "ok": 105,
    "total": 105,
    "bad_first10": [],
    "outside_zero_ok": 105,
    "active_exact_ok": 105
  },
  "arc_gen_full_diagnostic": {
    "ok": 262,
    "total": 262,
    "bad_first10": [],
    "outside_zero_ok": 262,
    "active_exact_ok": 262
  }
}


In [12]:
# Extra hidden-generalization diagnostics: shifted canvases and variable cell sizes.
def solve_symbolic_np(grid):
    g=np.array(grid, dtype=int)
    out=np.zeros_like(g)
    active=np.ones_like(g, dtype=bool)
    sep=(g==5)
    out[sep]=5
    rows=np.where(sep.all(axis=1))[0].tolist()
    cols=np.where(sep.all(axis=0))[0].tolist()
    row_starts=[0]+[r+1 for r in rows]
    row_ends=rows+[g.shape[0]]
    col_starts=[0]+[c+1 for c in cols]
    col_ends=cols+[g.shape[1]]
    cells=[]; max_count=0
    for rs,re in zip(row_starts,row_ends):
        for cs,ce in zip(col_starts,col_ends):
            if rs>=re or cs>=ce: continue
            sub=g[rs:re,cs:ce]
            nz=sub[(sub!=0)&(sub!=5)]
            cnt=int(nz.size)
            col=int(nz[0]) if cnt else 0
            cells.append((rs,re,cs,ce,cnt,col))
            max_count=max(max_count,cnt)
    for rs,re,cs,ce,cnt,col in cells:
        if cnt==max_count and cnt>0:
            out[rs:re,cs:ce]=col
    return out.tolist()

def make_lattice_case(block_h=3, block_w=3, rows=3, cols=3, color=4, seed=1):
    rng=np.random.default_rng(seed)
    gh=rows*block_h+(rows-1)
    gw=cols*block_w+(cols-1)
    g=np.zeros((gh,gw), dtype=int)
    for i in range(1,rows): g[i*block_h+i-1,:]=5
    for j in range(1,cols): g[:,j*block_w+j-1]=5
    for bi in range(rows):
        for bj in range(cols):
            rs=bi*(block_h+1); cs=bj*(block_w+1)
            # varied marker counts; never overwrite separators
            cnt=(bi*2+bj+seed) % max(1, min(block_h*block_w, 7))
            coords=[(r,c) for r in range(rs,rs+block_h) for c in range(cs,cs+block_w)]
            rng.shuffle(coords)
            for r,c in coords[:cnt]: g[r,c]=color
    return {'input':g.tolist(), 'output':solve_symbolic_np(g.tolist())}

synthetic=[]
for seed,(bh,bw,rr,cc) in enumerate([(2,2,3,3),(4,3,3,2),(5,4,2,3),(6,6,2,2)], start=10):
    synthetic.append(make_lattice_case(bh,bw,rr,cc, color=(seed%8)+1, seed=seed))
print('synthetic variable-lattice:', validate_examples(synthetic))

# Shift visible test inside 30x30 and validate tensor-exact zero padding.
shifted=[]
base=task['test'][0]
for off in [(2,3),(10,5),(18,18)]:
    if off[0]+len(base['input']) <= H and off[1]+len(base['input'][0]) <= W:
        shifted.append({'input':base['input'], 'output':base['output'], 'offset':off})
shift_ok=0
for ex in shifted:
    x=grid_to_tensor(ex['input'], offset=ex['offset'])
    y=sess.run(None, {'input':x})[0]
    pred=(y>0.5).astype(np.float32)
    exp=grid_to_tensor(ex['output'], offset=ex['offset'])
    shift_ok += int(np.array_equal(pred, exp))
print('shifted visible test:', shift_ok, '/', len(shifted))
assert shift_ok == len(shifted)

summary['synthetic_variable_lattice'] = validate_examples(synthetic)
summary['shifted_visible_test'] = {'ok':shift_ok, 'total':len(shifted)}
summary.update(summary_static)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

synthetic variable-lattice: {'ok': 4, 'total': 4, 'bad_first10': [], 'outside_zero_ok': 4, 'active_exact_ok': 4}
shifted visible test: 3 / 3


1387

In [13]:
with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
print('Wrote:', SUBMISSION_PATH)
print('Zip contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f'{TASK_ID}.onnx']

Wrote: /kaggle/working/submission.zip
Zip contents: ['task059.onnx']
